In [1]:
# Step 1: Import necessary libraries
import torch
import os
import shutil
from sklearn.model_selection import train_test_split
import xml.etree.ElementTree as ET
from transformers import DetrImageProcessor, DetrForObjectDetection
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import numpy as np
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

In [2]:
# Step 2: Define paths and dataset setup
DATASET_PATH = r'/home/ahabb/Downloads/fod-a/FullDatasetV.2.1-400x400'
FOLDERS = os.listdir(DATASET_PATH)
if 'All_Dataset_Utility_Files' in FOLDERS:
    FOLDERS.remove('All_Dataset_Utility_Files')
if 'classes.txt' in FOLDERS:
    FOLDERS.remove('classes.txt')
if 'fod_dataset.yaml' in FOLDERS:
    FOLDERS.remove('fod_dataset.yaml')
if 'train' in FOLDERS:
    FOLDERS.remove('train')
if 'test' in FOLDERS:
    FOLDERS.remove('test')
if 'val' in FOLDERS:
    FOLDERS.remove('val')


In [3]:
# Step 3: Data preparation function
def prepare_dataset(image_dir, annotations_dir, number=0):
    images = sorted([f for f in os.listdir(image_dir) if f.endswith('.PNG')])
    annotations = sorted([f for f in os.listdir(annotations_dir) if f.endswith('.xml')])

    # Pair images and annotations
    image_annotation_pairs = list(zip(images, annotations))
    train_val_split, test_split = train_test_split(image_annotation_pairs, test_size=0.3)
    train_split, val_split = train_test_split(train_val_split, test_size=0.2)

    splits = {'train': train_split, 'val': val_split, 'test': test_split}
    
    # Create directories and copy files
    for split_name, pairs in splits.items():
        for sub_dir in ['images', 'labels']:
            os.makedirs(os.path.join(DATASET_PATH, split_name, sub_dir), exist_ok=True)
        
        for img_name, ann_name in pairs:
            num = str(number)
            shutil.copy(
                os.path.join(image_dir, img_name), 
                os.path.join(DATASET_PATH, split_name, 'images', num + '.PNG')
            )
            shutil.copy(
                os.path.join(annotations_dir, ann_name), 
                os.path.join(DATASET_PATH, split_name, 'labels', num + '.xml')
            )
            number += 1
    
    return number


In [4]:
# Step 4: Process all folders and prepare data
number = 0
for folder in FOLDERS:
    IMAGE_DIR = os.path.join(DATASET_PATH, folder, 'frame')
    ANNOTATIONS_DIR = os.path.join(DATASET_PATH, folder, 'Annotations')
    number = prepare_dataset(IMAGE_DIR, ANNOTATIONS_DIR, number)


In [67]:
import os
import torch
from torch.utils.data import Dataset
from PIL import Image
from transformers import DetrImageProcessor

class FODDataset(Dataset):
    def __init__(self, img_folder, ann_folder, processor):
        self.img_folder = img_folder
        self.ann_folder = ann_folder
        self.processor = processor
        self.imgs = sorted(os.listdir(img_folder))
        self.anns = sorted(os.listdir(ann_folder))

    def __len__(self):
        return len(self.imgs)

    def __getitem__(self, idx):
        img_path = os.path.join(self.img_folder, self.imgs[idx])
        ann_path = os.path.join(self.ann_folder, self.anns[idx])
        image = Image.open(img_path).convert("RGB")

        # Attempt to parse annotations, handle any errors gracefully
        try:
            boxes, labels = self.parse_yolo_txt(ann_path, image.size)
            if len(boxes) == 0:
                raise ValueError("No valid boxes found.")
        except Exception as e:
            print(f"Error processing file {ann_path}: {e}")
            return None  # Skip this item or return a default value

        # Construct the COCO-style annotations
        annotations = []
        for box, label in zip(boxes, labels):
            x_min, y_min, x_max, y_max = box
            width = x_max - x_min
            height = y_max - y_min
            annotations.append({
                "bbox": [x_min, y_min, width, height],
                "category_id": int(label),
                "area": width * height,
                "iscrowd": 0
            })

        # Image ID is just the index in this case
        target = {
            "image_id": idx,
            "annotations": annotations
        }

        # Preprocess the image and annotations
        encoding = self.processor(images=image, annotations=[target], return_tensors="pt")
        return encoding

    def parse_yolo_txt(self, txt_path, image_size):
        width, height = image_size
        boxes = []
        labels = []

        try:
            with open(txt_path, 'r') as f:
                for line in f.readlines():
                    parts = line.strip().split()
                    if len(parts) != 5:  # Ensure correct YOLO format
                        continue

                    class_id = int(parts[0])
                    x_center = float(parts[1]) * width
                    y_center = float(parts[2]) * height
                    box_width = float(parts[3]) * width
                    box_height = float(parts[4]) * height

                    # Convert to [x_min, y_min, x_max, y_max]
                    x_min = x_center - box_width / 2
                    y_min = y_center - box_height / 2
                    x_max = x_center + box_width / 2
                    y_max = y_center + box_height / 2

                    boxes.append([x_min, y_min, x_max, y_max])
                    labels.append(class_id)

        except Exception as e:
            print(f"Error reading {txt_path}: {e}")

        return torch.tensor(boxes, dtype=torch.float32), torch.tensor(labels, dtype=torch.int64)


In [68]:
import os
import glob
import torch
from torch.utils.data import Dataset
from PIL import Image
from transformers import DetrImageProcessor

class FODDataset(Dataset):
    def __init__(self, img_folder, ann_folder, processor):
        self.img_folder = img_folder
        self.ann_folder = ann_folder
        self.processor = processor
        self.imgs = sorted(os.listdir(img_folder))
        self.anns = sorted(glob.glob(os.path.join(ann_folder, '*.txt')))  # Use glob to get .txt files

        # Ensure that the images and annotations match
        assert len(self.imgs) == len(self.anns), "Mismatch between number of images and annotations"

    def __len__(self):
        return len(self.imgs)

    def __getitem__(self, idx):
        img_path = os.path.join(self.img_folder, self.imgs[idx])
        ann_path = self.anns[idx]

        image = Image.open(img_path).convert("RGB")

        # Attempt to parse annotations, handle any errors gracefully
        try:
            boxes, labels = self.parse_yolo_txt(ann_path, image.size)
            if len(boxes) == 0:
                raise ValueError("No valid boxes found.")
        except Exception as e:
            print(f"Error processing file {ann_path}: {e}")
            return self.__getitem__((idx + 1) % len(self))  # Try the next item, wrap around

        # Construct the COCO-style annotations
        annotations = []
        for box, label in zip(boxes, labels):
            x_min, y_min, x_max, y_max = box
            width = x_max - x_min
            height = y_max - y_min
            annotations.append({
                "bbox": [x_min, y_min, width, height],
                "category_id": int(label),
                "area": width * height,
                "iscrowd": 0
            })

        # Image ID is just the index in this case
        target = {
            "image_id": idx,
            "annotations": annotations
        }

        # Preprocess the image and annotations
        encoding = self.processor(images=image, annotations=[target], return_tensors="pt")
        return encoding

    def parse_yolo_txt(self, txt_path, image_size):
        width, height = image_size
        boxes = []
        labels = []

        try:
            with open(txt_path, 'r') as f:
                for line in f.readlines():
                    parts = line.strip().split()
                    if len(parts) != 5:  # Ensure correct YOLO format
                        continue

                    class_id = int(parts[0])
                    x_center = float(parts[1]) * width
                    y_center = float(parts[2]) * height
                    box_width = float(parts[3]) * width
                    box_height = float(parts[4]) * height

                    # Convert to [x_min, y_min, x_max, y_max]
                    x_min = x_center - box_width / 2
                    y_min = y_center - box_height / 2
                    x_max = x_center + box_width / 2
                    y_max = y_center + box_height / 2

                    boxes.append([x_min, y_min, x_max, y_max])
                    labels.append(class_id)

        except Exception as e:
            print(f"Error reading {txt_path}: {e}")

        return torch.tensor(boxes, dtype=torch.float32), torch.tensor(labels, dtype=torch.int64)


In [69]:
# Step 6: Load the classes from classes.txt and create a mapping
CLASS_FILE = os.path.join(DATASET_PATH, 'classes.txt')
with open(CLASS_FILE, 'r') as f:
    classes = [line.strip() for line in f.readlines()]
class_name_to_id = {name: idx for idx, name in enumerate(classes)}


In [52]:
# Step 7: Initialize the model and processor
processor = DetrImageProcessor.from_pretrained("facebook/detr-resnet-50")
model = DetrForObjectDetection.from_pretrained("facebook/detr-resnet-50")


Some weights of the model checkpoint at facebook/detr-resnet-50 were not used when initializing DetrForObjectDetection: ['model.backbone.conv_encoder.model.layer1.0.downsample.1.num_batches_tracked', 'model.backbone.conv_encoder.model.layer2.0.downsample.1.num_batches_tracked', 'model.backbone.conv_encoder.model.layer3.0.downsample.1.num_batches_tracked', 'model.backbone.conv_encoder.model.layer4.0.downsample.1.num_batches_tracked']
- This IS expected if you are initializing DetrForObjectDetection from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DetrForObjectDetection from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [70]:
# Step 8: Create datasets and dataloaders
train_dataset = FODDataset(
    img_folder=os.path.join(DATASET_PATH, 'train/images'),
    ann_folder=os.path.join(DATASET_PATH, 'train/labels'),
    processor=processor
)
val_dataset = FODDataset(
    img_folder=os.path.join(DATASET_PATH, 'val/images'),
    ann_folder=os.path.join(DATASET_PATH, 'val/labels'),
    processor=processor
)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=4)


In [75]:
train_loader.shape

AttributeError: 'DataLoader' object has no attribute 'shape'

In [74]:
# Fetch one batch from the train_loader
for images, labels in train_loader:
    print(f'Image batch shape: {images.shape}')
    break  # Only need the first batch


ValueError: too many values to unpack (expected 2)

In [71]:
# Step 9: Define training loop with loss and accuracy tracking
from transformers import AdamW
from tqdm.notebook import tqdm

# Set up optimizer and device
optimizer = AdamW(model.parameters(), lr=1e-5)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Initialize tracking variables
train_losses = []
val_losses = []
val_accuracies = []

# Training function
def train_one_epoch(model, data_loader, optimizer):
    model.train()
    total_loss = 0
    for batch in tqdm(data_loader):
        pixel_values = batch["pixel_values"].to(device)
        labels = [{k: v.to(device) for k, v in t.items()} for t in batch["labels"]]

        outputs = model(pixel_values=pixel_values, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        total_loss += loss.item()
    
    return total_loss / len(data_loader)

# Evaluation function with accuracy calculation
def evaluate(model, data_loader):
    model.eval()
    total_loss = 0
    true_labels = []
    pred_labels = []

    with torch.no_grad():
        for batch in tqdm(data_loader):
            pixel_values = batch["pixel_values"].to(device)
            labels = [{k: v.to(device) for k, v in t.items()} for t in batch["labels"]]
            
            outputs = model(pixel_values=pixel_values, labels=labels)
            loss = outputs.loss
            total_loss += loss.item()
            
            # Extract true and predicted labels
            for label in labels:
                true_labels.extend(label['labels'].cpu().numpy())
            pred_logits = outputs.logits.argmax(-1).cpu().numpy()
            pred_labels.extend(pred_logits)
    
    avg_loss = total_loss / len(data_loader)
    accuracy = np.mean(np.array(true_labels) == np.array(pred_labels))
    return avg_loss, accuracy, true_labels, pred_labels


/home/ahabb/anaconda3/lib/python3.12/site-packages/transformers/optimization.py:591: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


In [72]:
# Step 10: Train the model and evaluate on validation set
num_epochs = 5  # Adjust as needed
for epoch in range(num_epochs):
    train_loss = train_one_epoch(model, train_loader, optimizer)
    val_loss, val_accuracy, true_labels, pred_labels = evaluate(model, val_loader)
    
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    val_accuracies.append(val_accuracy)
    
    print(f"Epoch {epoch + 1}/{num_epochs}, Training Loss: {train_loss:.4f}, Validation Loss: {val_loss:.4f}, Validation Accuracy: {val_accuracy:.4f}")


  0%|          | 0/4715 [00:00<?, ?it/s]

ValueError: too many values to unpack (expected 4)

In [ ]:
# Step 11: Plot Training and Validation Loss
plt.figure(figsize=(10, 5))
plt.plot(range(1, num_epochs + 1), train_losses, label='Training Loss')
plt.plot(range(1, num_epochs + 1), val_losses, label='Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.title('Training and Validation Loss')
plt.legend()
plt.show()

# Step 12: Plot Confusion Matrix for the Validation Set
cm = confusion_matrix(true_labels, pred_labels, labels=list(range(len(classes))))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=classes)
fig, ax = plt.subplots(figsize=(15, 15))
disp.plot(ax=ax, cmap='Blues', xticks_rotation='vertical')
plt.title('Confusion Matrix')
plt.show()